In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# GOLD DIMENSIONS - CREATION EN UNE SEULE CELLULE
# ============================================================


# ============================================================
# 1. DIM_STATION
# Source : sncf_silver.stops
# PK technique : station_sk
# Business key : stop_id
# ============================================================

df_station = (
    spark.table("workspace.sncf_silver.stops")
    .filter(F.col("stop_id").isNotNull())
    .dropDuplicates(["stop_id"])
    .withColumn(
        "station_sk",
        F.row_number().over(Window.orderBy("stop_id"))
    )
    .select(
        "station_sk",
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon",
        "location_type",
        "parent_station"
    )
)

(
    df_station.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_gold.dim_station")
)


# ============================================================
# 2. DIM_AGENCY
# Source : sncf_silver.agency
# PK technique : agency_sk
# Business key : agency_id
# ============================================================

df_agency = (
    spark.table("workspace.sncf_silver.agency")
    .filter(F.col("agency_id").isNotNull())
    .dropDuplicates(["agency_id"])
    .withColumn(
        "agency_sk",
        F.row_number().over(Window.orderBy("agency_id"))
    )
    .select(
        "agency_sk",
        "agency_id",
        "agency_name",
        "agency_url",
        "agency_timezone"
    )
)

(
    df_agency.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_gold.dim_agency")
)


# ============================================================
# 3. DIM_ROUTE
# Source : sncf_silver.routes
# PK technique : route_sk
# FK : agency_sk
# ============================================================

df_routes_source = (
    spark.table("workspace.sncf_silver.routes")
    .filter(F.col("route_id").isNotNull())
    .dropDuplicates(["route_id"])
)

df_route = (
    df_routes_source
    .join(
        df_agency.select("agency_id", "agency_sk"),
        on="agency_id",
        how="left"
    )
    .withColumn(
        "route_sk",
        F.row_number().over(Window.orderBy("route_id"))
    )
    .select(
        "route_sk",
        "route_id",
        "agency_sk",
        "route_short_name",
        "route_long_name",
        "route_type",
        "route_color",
        "route_text_color"
    )
)

(
    df_route.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_gold.dim_route")
)


# ============================================================
# 4. DIM_SERVICE
# Source : sncf_silver.trips
# PK technique : service_sk
# Business key : service_id
# ============================================================

df_service = (
    spark.table("workspace.sncf_silver.trips")
    .select("service_id")
    .filter(F.col("service_id").isNotNull())
    .distinct()
    .withColumn(
        "service_sk",
        F.row_number().over(Window.orderBy("service_id"))
    )
    .select(
        "service_sk",
        "service_id"
    )
)

(
    df_service.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_gold.dim_service")
)


# ============================================================
# 5. DIM_DATE
# Source : sncf_silver.calendar_dates
# PK : date_sk
# ============================================================

df_date = (
    spark.table("workspace.sncf_silver.calendar_dates")
    .select("date")
    .filter(F.col("date").isNotNull())
    .distinct()

    # YYYY-MM-DD -> YYYYMMDD
    .withColumn(
        "date_sk",
        F.date_format("date", "yyyyMMdd").cast("int")
    )

    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("week_of_year", F.weekofyear("date"))

    .withColumn(
        "is_weekend",
        F.when(
            F.dayofweek("date").isin(1, 7),
            True
        ).otherwise(False)
    )

    .select(
        "date_sk",
        "date",
        "year",
        "quarter",
        "month",
        "day",
        "day_of_week",
        "week_of_year",
        "is_weekend"
    )
)

(
    df_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_gold.dim_date")
)


# ============================================================
# 6. VALIDATION
# ============================================================

print("===== GOLD DIMENSIONS CRÉÉES =====")

for table in [
    "dim_station",
    "dim_agency",
    "dim_route",
    "dim_service",
    "dim_date"
]:
    df = spark.table(f"workspace.sncf_gold.{table}")

    print(
        table,
        "=>",
        df.count(),
        "lignes"
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


===== GOLD DIMENSIONS CRÉÉES =====
dim_station => 8678 lignes
dim_agency => 6 lignes
dim_route => 697 lignes
dim_service => 6693 lignes
dim_date => 179 lignes


In [0]:
from pyspark.sql import functions as F

# ============================================================
# GOLD FACT_STOP_TIMES
# ============================================================

# ------------------------------------------------------------
# 1. SOURCES SILVER
# ------------------------------------------------------------

df_stop_times = spark.table("workspace.sncf_silver.stop_times")

df_trips = (
    spark.table("workspace.sncf_silver.trips")
    .select(
        "trip_id",
        "route_id",
        "service_id"
    )
)

df_calendar_dates = (
    spark.table("workspace.sncf_silver.calendar_dates")
    .filter(F.col("exception_type") == 1)
    .select(
        "service_id",
        "date"
    )
)


# ------------------------------------------------------------
# 2. DIMENSIONS GOLD
# ------------------------------------------------------------

df_dim_station = (
    spark.table("workspace.sncf_gold.dim_station")
    .select(
        "station_sk",
        "stop_id"
    )
)

df_dim_route = (
    spark.table("workspace.sncf_gold.dim_route")
    .select(
        "route_sk",
        "route_id"
    )
)

df_dim_service = (
    spark.table("workspace.sncf_gold.dim_service")
    .select(
        "service_sk",
        "service_id"
    )
)

df_dim_date = (
    spark.table("workspace.sncf_gold.dim_date")
    .select(
        "date_sk",
        "date"
    )
)


# ============================================================
# 3. ENRICHISSEMENT STOP_TIMES AVEC TRIPS
# ============================================================

df_fact = (
    df_stop_times

    # stop_times -> trips
    .join(
        df_trips,
        on="trip_id",
        how="inner"
    )

    # trips.service_id -> calendar_dates
    # On ne garde que exception_type = 1 :
    # service actif / ajouté à cette date
    .join(
        df_calendar_dates,
        on="service_id",
        how="inner"
    )
)


# ============================================================
# 4. RÉCUPÉRATION DES SURROGATE KEYS
# ============================================================

df_fact = (
    df_fact

    # stop_id -> station_sk
    .join(
        df_dim_station,
        on="stop_id",
        how="left"
    )

    # route_id -> route_sk
    .join(
        df_dim_route,
        on="route_id",
        how="left"
    )

    # service_id -> service_sk
    .join(
        df_dim_service,
        on="service_id",
        how="left"
    )

    # date -> date_sk
    .join(
        df_dim_date,
        on="date",
        how="left"
    )
)


# ============================================================
# 5. SÉLECTION DU MODÈLE GOLD
# ============================================================

df_fact_stop_times = (
    df_fact
    .select(
        # Foreign Keys
        "station_sk",
        "route_sk",
        "service_sk",
        "date_sk",

        # Identifiants métier
        "trip_id",
        "stop_id",
        "route_id",
        "service_id",

        # Grain
        "stop_sequence",

        # Informations horaires
        "arrival_time",
        "departure_time",

        # Données métier GTFS
        "pickup_type",
        "drop_off_type",

        # Date réelle du service
        "date"
    )

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)


# ============================================================
# 6. CONTRÔLES FK
# ============================================================

print("===== CONTROLES FACT_STOP_TIMES =====")

print("Nombre de lignes :", df_fact_stop_times.count())

print(
    "station_sk NULL :",
    df_fact_stop_times
    .filter(F.col("station_sk").isNull())
    .count()
)

print(
    "route_sk NULL :",
    df_fact_stop_times
    .filter(F.col("route_sk").isNull())
    .count()
)

print(
    "service_sk NULL :",
    df_fact_stop_times
    .filter(F.col("service_sk").isNull())
    .count()
)

print(
    "date_sk NULL :",
    df_fact_stop_times
    .filter(F.col("date_sk").isNull())
    .count()
)


# ============================================================
# 7. CONTRÔLE DU GRAIN
# Une ligne doit être unique pour :
# date + trip + stop_sequence
# ============================================================

duplicates = (
    df_fact_stop_times
    .groupBy(
        "date",
        "trip_id",
        "stop_sequence"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Doublons du grain :",
    duplicates.count()
)

display(df_fact_stop_times.limit(20))

===== CONTROLES FACT_STOP_TIMES =====
Nombre de lignes : 8517376
station_sk NULL : 0
route_sk NULL : 0
service_sk NULL : 0
date_sk NULL : 0
Doublons du grain : 0


station_sk,route_sk,service_sk,date_sk,trip_id,stop_id,route_id,service_id,stop_sequence,arrival_time,departure_time,pickup_type,drop_off_type,date,_gold_processed_at
5892,342,1,20261030,OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000001,0,19:54:00,19:54:00,0,1,2026-10-30,2026-09-04T11:46:02.857Z
5893,342,1,20261030,OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000001,1,19:59:00,19:59:00,1,0,2026-10-30,2026-09-04T11:46:02.857Z
5893,342,2,20261212,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000002,0,17:23:00,17:23:00,0,1,2026-12-12,2026-09-04T11:46:02.857Z
5892,342,2,20261212,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000002,1,17:28:00,17:28:00,1,0,2026-12-12,2026-09-04T11:46:02.857Z
5893,342,3,20261211,OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000003,0,22:10:00,22:10:00,0,1,2026-12-11,2026-09-04T11:46:02.857Z
5892,342,3,20261211,OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000003,1,22:15:00,22:15:00,1,0,2026-12-11,2026-09-04T11:46:02.857Z
5893,342,4,20261207,OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000004,0,05:55:00,05:55:00,0,1,2026-12-07,2026-09-04T11:46:02.857Z
5892,342,4,20261207,OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000004,1,06:00:00,06:00:00,1,0,2026-12-07,2026-09-04T11:46:02.857Z
5893,342,5,20261007,OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000005,0,10:40:00,10:40:00,0,1,2026-10-07,2026-09-04T11:46:02.857Z
5892,342,5,20261007,OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000005,1,10:46:00,10:46:00,1,0,2026-10-07,2026-09-04T11:46:02.857Z


In [0]:
# ============================================================
# ECRITURE FACT_STOP_TIMES
# ============================================================

(
    df_fact_stop_times.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.sncf_gold.fact_stop_times"
    )
)


# ============================================================
# VALIDATION
# ============================================================

df_result = spark.table(
    "workspace.sncf_gold.fact_stop_times"
)

print(
    "FACT_STOP_TIMES créée :",
    df_result.count(),
    "lignes"
)

df_result.printSchema()

display(df_result.limit(20))

FACT_STOP_TIMES créée : 8517376 lignes
root
 |-- station_sk: integer (nullable = true)
 |-- route_sk: integer (nullable = true)
 |-- service_sk: integer (nullable = true)
 |-- date_sk: integer (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- service_id: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- pickup_type: integer (nullable = true)
 |-- drop_off_type: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- _gold_processed_at: timestamp (nullable = true)



station_sk,route_sk,service_sk,date_sk,trip_id,stop_id,route_id,service_id,stop_sequence,arrival_time,departure_time,pickup_type,drop_off_type,date,_gold_processed_at
5892,342,1,20261030,OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000001,0,19:54:00,19:54:00,0,1,2026-10-30,2026-09-04T11:46:26.743Z
5893,342,1,20261030,OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000001,1,19:59:00,19:59:00,1,0,2026-10-30,2026-09-04T11:46:26.743Z
5893,342,2,20261212,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000002,0,17:23:00,17:23:00,0,1,2026-12-12,2026-09-04T11:46:26.743Z
5892,342,2,20261212,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000002,1,17:28:00,17:28:00,1,0,2026-12-12,2026-09-04T11:46:26.743Z
5893,342,3,20261211,OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000003,0,22:10:00,22:10:00,0,1,2026-12-11,2026-09-04T11:46:26.743Z
5892,342,3,20261211,OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000003,1,22:15:00,22:15:00,1,0,2026-12-11,2026-09-04T11:46:26.743Z
5893,342,4,20261207,OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000004,0,05:55:00,05:55:00,0,1,2026-12-07,2026-09-04T11:46:26.743Z
5892,342,4,20261207,OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000004,1,06:00:00,06:00:00,1,0,2026-12-07,2026-09-04T11:46:26.743Z
5893,342,5,20261007,OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,StopPoint:OCENavette-87571240,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000005,0,10:40:00,10:40:00,0,1,2026-10-07,2026-09-04T11:46:26.743Z
5892,342,5,20261007,OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,StopPoint:OCENavette-87571000,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000005,1,10:46:00,10:46:00,1,0,2026-10-07,2026-09-04T11:46:26.743Z


In [0]:
#1 row = 1 transfer rule between one origin stop and one destination stop
from pyspark.sql import functions as F

# ============================================================
# GOLD FACT_TRANSFERS
# ============================================================

# ------------------------------------------------------------
# 1. SOURCE SILVER
# ------------------------------------------------------------

df_transfers = spark.table(
    "workspace.sncf_silver.transfers"
)

# ------------------------------------------------------------
# 2. DIMENSION STATION
# We use dim_station twice:
# - once for the origin
# - once for the destination
# ------------------------------------------------------------

df_station_from = (
    spark.table("workspace.sncf_gold.dim_station")
    .select(
        F.col("stop_id").alias("from_stop_id"),
        F.col("station_sk").alias("from_station_sk")
    )
)

df_station_to = (
    spark.table("workspace.sncf_gold.dim_station")
    .select(
        F.col("stop_id").alias("to_stop_id"),
        F.col("station_sk").alias("to_station_sk")
    )
)

# ------------------------------------------------------------
# 3. JOIN TO GET THE GOLD FOREIGN KEYS
# ------------------------------------------------------------

df_fact_transfers = (
    df_transfers

    .join(
        df_station_from,
        on="from_stop_id",
        how="left"
    )

    .join(
        df_station_to,
        on="to_stop_id",
        how="left"
    )

    .select(
        # Foreign Keys
        "from_station_sk",
        "to_station_sk",

        # Business keys kept for traceability
        "from_stop_id",
        "to_stop_id",

        # Transfer information
        "transfer_type",
        "min_transfer_time"
    )

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

# ============================================================
# 4. DATA QUALITY
# ============================================================

print("===== CONTROLES FACT_TRANSFERS =====")

print(
    "Nombre de lignes :",
    df_fact_transfers.count()
)

print(
    "from_station_sk NULL :",
    df_fact_transfers
    .filter(F.col("from_station_sk").isNull())
    .count()
)

print(
    "to_station_sk NULL :",
    df_fact_transfers
    .filter(F.col("to_station_sk").isNull())
    .count()
)

# Logical grain check
duplicates = (
    df_fact_transfers
    .groupBy(
        "from_stop_id",
        "to_stop_id",
        "transfer_type"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Doublons du grain :",
    duplicates.count()
)

display(df_fact_transfers.limit(20))

===== CONTROLES FACT_TRANSFERS =====
Nombre de lignes : 0
from_station_sk NULL : 0
to_station_sk NULL : 0
Doublons du grain : 0


from_station_sk,to_station_sk,from_stop_id,to_stop_id,transfer_type,min_transfer_time,_gold_processed_at


In [0]:
(
    df_fact_transfers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.sncf_gold.fact_transfers"
    )
)

df_result = spark.table(
    "workspace.sncf_gold.fact_transfers"
)

print(
    "FACT_TRANSFERS créée :",
    df_result.count(),
    "lignes"
)

df_result.printSchema()
display(df_result.limit(20))

FACT_TRANSFERS créée : 0 lignes
root
 |-- from_station_sk: integer (nullable = true)
 |-- to_station_sk: integer (nullable = true)
 |-- from_stop_id: string (nullable = true)
 |-- to_stop_id: string (nullable = true)
 |-- transfer_type: integer (nullable = true)
 |-- min_transfer_time: integer (nullable = true)
 |-- _gold_processed_at: timestamp (nullable = true)



from_station_sk,to_station_sk,from_stop_id,to_stop_id,transfer_type,min_transfer_time,_gold_processed_at


In [0]:
from pyspark.sql import functions as F

print("====================================================")
print("     VALIDATION COMPLETE DU MODELE GOLD")
print("====================================================")


# ============================================================
# 1. LOAD GOLD TABLES
# ============================================================

dim_station = spark.table("workspace.sncf_gold.dim_station")
dim_agency = spark.table("workspace.sncf_gold.dim_agency")
dim_route = spark.table("workspace.sncf_gold.dim_route")
dim_service = spark.table("workspace.sncf_gold.dim_service")
dim_date = spark.table("workspace.sncf_gold.dim_date")

fact_stop_times = spark.table("workspace.sncf_gold.fact_stop_times")
fact_transfers = spark.table("workspace.sncf_gold.fact_transfers")


# ============================================================
# 2. CHECK PRIMARY / SURROGATE KEYS
# ============================================================

pk_checks = [
    ("dim_station", dim_station, "station_sk"),
    ("dim_agency", dim_agency, "agency_sk"),
    ("dim_route", dim_route, "route_sk"),
    ("dim_service", dim_service, "service_sk"),
    ("dim_date", dim_date, "date_sk"),
]

for table_name, df, pk in pk_checks:

    null_count = df.filter(F.col(pk).isNull()).count()

    duplicate_count = (
        df.groupBy(pk)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name} | {pk} NULL = {null_count} | "
        f"doublons = {duplicate_count}"
    )


# ============================================================
# 3. DIM_ROUTE -> DIM_AGENCY
# ============================================================

invalid_agency_fk = (
    dim_route
    .select("agency_sk")
    .filter(F.col("agency_sk").isNotNull())
    .distinct()
    .join(
        dim_agency.select("agency_sk"),
        on="agency_sk",
        how="left_anti"
    )
)

print(
    "\nFK dim_route.agency_sk invalide :",
    invalid_agency_fk.count()
)


# ============================================================
# 4. FACT_STOP_TIMES FOREIGN KEYS
# ============================================================

invalid_station = (
    fact_stop_times
    .select("station_sk")
    .distinct()
    .join(
        dim_station.select("station_sk"),
        "station_sk",
        "left_anti"
    )
)

invalid_route = (
    fact_stop_times
    .select("route_sk")
    .distinct()
    .join(
        dim_route.select("route_sk"),
        "route_sk",
        "left_anti"
    )
)

invalid_service = (
    fact_stop_times
    .select("service_sk")
    .distinct()
    .join(
        dim_service.select("service_sk"),
        "service_sk",
        "left_anti"
    )
)

invalid_date = (
    fact_stop_times
    .select("date_sk")
    .distinct()
    .join(
        dim_date.select("date_sk"),
        "date_sk",
        "left_anti"
    )
)

print("\n===== FACT_STOP_TIMES =====")

print(
    "station_sk invalides :",
    invalid_station.count()
)

print(
    "route_sk invalides :",
    invalid_route.count()
)

print(
    "service_sk invalides :",
    invalid_service.count()
)

print(
    "date_sk invalides :",
    invalid_date.count()
)


# ============================================================
# 5. FACT_STOP_TIMES GRAIN
# Grain:
# date + trip_id + stop_sequence
# ============================================================

fact_stop_times_duplicates = (
    fact_stop_times
    .groupBy(
        "date_sk",
        "trip_id",
        "stop_sequence"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Doublons grain fact_stop_times :",
    fact_stop_times_duplicates.count()
)


# ============================================================
# 6. FACT_TRANSFERS FOREIGN KEYS
# ============================================================

invalid_from_station = (
    fact_transfers
    .select(
        F.col("from_station_sk").alias("station_sk")
    )
    .distinct()
    .join(
        dim_station.select("station_sk"),
        "station_sk",
        "left_anti"
    )
)

invalid_to_station = (
    fact_transfers
    .select(
        F.col("to_station_sk").alias("station_sk")
    )
    .distinct()
    .join(
        dim_station.select("station_sk"),
        "station_sk",
        "left_anti"
    )
)

print("\n===== FACT_TRANSFERS =====")

print(
    "from_station_sk invalides :",
    invalid_from_station.count()
)

print(
    "to_station_sk invalides :",
    invalid_to_station.count()
)


# ============================================================
# 7. FACT_TRANSFERS GRAIN
# ============================================================

fact_transfers_duplicates = (
    fact_transfers
    .groupBy(
        "from_station_sk",
        "to_station_sk",
        "transfer_type"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Doublons grain fact_transfers :",
    fact_transfers_duplicates.count()
)


# ============================================================
# 8. ROW COUNTS
# ============================================================

print("\n===== VOLUMES GOLD =====")

for table in [
    "dim_station",
    "dim_agency",
    "dim_route",
    "dim_service",
    "dim_date",
    "fact_stop_times",
    "fact_transfers"
]:

    count = spark.table(
        f"workspace.sncf_gold.{table}"
    ).count()

    print(
        table,
        "=>",
        count,
        "lignes"
    )


print("\n====================================================")
print("Validation terminée.")
print("Idéalement toutes les FK invalides et doublons = 0.")
print("====================================================")

     VALIDATION COMPLETE DU MODELE GOLD
dim_station | station_sk NULL = 0 | doublons = 0
dim_agency | agency_sk NULL = 0 | doublons = 0
dim_route | route_sk NULL = 0 | doublons = 0
dim_service | service_sk NULL = 0 | doublons = 0
dim_date | date_sk NULL = 0 | doublons = 0

FK dim_route.agency_sk invalide : 0

===== FACT_STOP_TIMES =====
station_sk invalides : 0
route_sk invalides : 0
service_sk invalides : 0
date_sk invalides : 0
Doublons grain fact_stop_times : 0

===== FACT_TRANSFERS =====
from_station_sk invalides : 0
to_station_sk invalides : 0
Doublons grain fact_transfers : 0

===== VOLUMES GOLD =====
dim_station => 8678 lignes
dim_agency => 6 lignes
dim_route => 697 lignes
dim_service => 6693 lignes
dim_date => 179 lignes
fact_stop_times => 8517376 lignes
fact_transfers => 0 lignes

Validation terminée.
Idéalement toutes les FK invalides et doublons = 0.
